# Part 5 of 5: Generalisation, Confidence & Discussion

COMP7015 Group Project – Topic 1: Plant Disease Classification. This notebook covers Sections 15–17: photo series that were left out of training, field photos (PlantDoc), confidence and abstention, and the discussion of all results.

The notebooks run in order: `01_data` → `02_models_and_tuning` → `03_test_evaluation` → `04_background_bias` → `05_generalisation`. The shared code lives in the `plantdisease/` package. Finished training runs and predictions are saved under `OUT_DIR/runs/` and reused, so this notebook can be re-run on its own once the earlier ones have run.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from plantdisease.backgrounds import make_test_variants
from plantdisease.config import CFG, CLASSES, SEEDS, SHORT_NAMES, out_path
from plantdisease.data import (
    IMAGE_SUFFIXES,
    border_brightness,
    build_cache,
    chunked,
    prepare,
    sparse_checkout,
)
from plantdisease.evaluate import (
    countermeasure_verdicts,
    expected_calibration_error,
    per_class_recall,
    risk_coverage,
    shift_table,
)
from plantdisease.experiments import countermeasure_names, load_final_models
from plantdisease.plots import finish, grouped_bars
from plantdisease.train import predict_labels, predict_probs

# selected on the validation set (notebook 02)
BEST_CNN, BEST_RESNET, BEST_VIT = load_final_models()
FINAL_MODELS = [BEST_CNN, BEST_RESNET, BEST_VIT]
data = prepare()  # leaf-group split of notebook 01, rebuilt from the image cache
BASES = (BEST_CNN, BEST_RESNET)  # the models with a countermeasure (Section 14)
compared = [name for base in BASES for name in (base, *countermeasure_names(base))]
shift_models = list(dict.fromkeys([*FINAL_MODELS, *compared]))
TEST_IMAGES = data.images[data.test_idx]

## 15. Does It Hold for a Different Photo Series?

Sections 13 and 14 change only the background of the same leaf. Here the photo changes as a whole: background, light, camera, the individual plants (the crop is always tomato) and the disease stage differ together. The photo series that were left out of training (Section 3) are leaves of the same classes that no model has seen. We evaluate all trained models on them (15.1), and on photos taken outside the lab (15.2), without retraining. A drop shows that a model does not transfer, not why. Noyan (2022) argues that the bias of PlantVillage comes from how the photos were captured (camera, light and background together), so removing the background alone does not remove it. Sections 13 and 14 test the background; this section tests whether the models also cope with other series and with field photos.

Early blight has only one series and does not occur in this test (a prediction of early blight counts as an error). The score is the macro-F1 over the classes that occur, next to the clean test of the same classes. The intervals are bootstrap intervals over leaf groups (Section 10.1); for series without official ids the groups come from the file name, and photos without leaf information are groups of their own.


### 15.1 Series that were left out of training


In [ ]:
other_df = data.other_df
SERIES_IMAGES = build_cache(other_df["path"], "other_series")
y_series = other_df["label"].to_numpy()
present = np.unique(y_series)  # classes that occur in the left-out series
other_df["border_brightness"] = chunked(border_brightness, SERIES_IMAGES).numpy()
print(
    f"{len(other_df)} photos of {other_df.series.nunique()} left-out series | "
    f"classes: {', '.join(SHORT_NAMES[c] for c in present)}"
)
other_df.groupby(["class_name", "series"]).agg(
    photos=("path", "size"),
    leaf_groups=("group", "nunique"),
    border_brightness=("border_brightness", "mean"),
).round(0)

In [ ]:
def shift_plot(results, *, clean_col, shift_col, seed_scores, ylabel, title, file_name):
    # Clean test vs. shifted photos per model (bars: mean, dots: seeds of the shifted photos).

    _, ax = plt.subplots(figsize=(11, 3.8))
    grouped_bars(
        ax,
        [n.replace(" + ", "\n+ ") for n in shift_models],
        ["clean test", shift_col],
        [results[clean_col], results[shift_col]],
        seeds=[None, [seed_scores[n] for n in shift_models]],
    )
    ax.tick_params(axis="x", labelsize=7)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1.05)
    ax.set_title(title)
    ax.legend(loc="lower left", fontsize=8)
    ax.grid(axis="y", alpha=0.3)
    finish(file_name)


series_preds = predict_labels(
    shift_models, {"test": TEST_IMAGES, "series": SERIES_IMAGES}
)
series_results, series_seed, series_boot = shift_table(
    shift_models,
    series_preds,
    "series",
    y=y_series,
    groups=other_df["group"].to_numpy(),
    y_clean=data.y_test,
    columns=(
        "clean test (same classes)",
        "left-out series",
        "95% interval (series leaves)",
    ),
)
series_results.to_csv(out_path("series_shift.csv"))
shift_plot(
    series_results,
    clean_col="clean test (same classes)",
    shift_col="left-out series",
    seed_scores=series_seed,
    ylabel="macro-F1 over the classes of the series test",
    title="Clean test and left-out photo series (bars: mean, dots: seeds)",
    file_name="series_shift.png",
)
series_results.round(3)

In [ ]:
# Recall per class and series, mean over the seeds
series_recall = pd.DataFrame(
    {
        name: {
            key: np.mean(
                [
                    (
                        series_preds[(name, seed, "series")][rows] == y_series[rows]
                    ).mean()
                    for seed in SEEDS
                ]
            )
            for key, rows in other_df.groupby(["class_name", "series"]).indices.items()
        }
        for name in shift_models
    }
)
series_recall.index.names = ["class", "series"]
series_recall.to_csv(out_path("series_shift_recall.csv"))
series_recall.round(3)

In [ ]:
series_verdicts = countermeasure_verdicts(
    BASES, series_seed, series_boot, "95% interval (series leaves)"
)
series_verdicts.to_csv(out_path("series_shift_verdicts.csv"), index=False)
series_verdicts.round(3)

#### Reading the series test

- The **drop** (clean test minus left-out series) mixes all differences between the series.
- The **recall per series** shows where the models break. The series differ, for example in background brightness (the dark `GHLB_PS`).
- If the countermeasure helps here, its gain is not limited to artificial backgrounds.
- Limits: early blight is missing, the series differ strongly in size, and the leaf groups of the greenhouse series come from file names.


### 15.2 Field photos (PlantDoc)

PlantDoc (Singh et al., 2020) collects photos of plants outside the lab: internet photos with several leaves, hands, soil and varying light. We use its cropped version, the four folders that match our classes (`Tomato leaf` is the healthy class) and **all** their photos as an external test, without retraining. The labels come from web sources and were only checked roughly, so the scores are indicative. PlantDoc has no leaf ids, so every photo counts as its own group in the bootstrap.


In [ ]:
FIELD, FIELD_ROOT = CFG["field_photos"], CFG["field_root"]
field_folders = [
    f"{s}/{n}" for s in ("train", "test") for n in FIELD["folders"].values()
]
FIELD_AVAILABLE = sparse_checkout(FIELD["url"], FIELD_ROOT, field_folders)
print("PlantDoc photos available:", FIELD_AVAILABLE)

In [ ]:
if FIELD_AVAILABLE:
    field_records = []
    for label, class_name in enumerate(CLASSES):
        # the authors' train/test split is not used: all photos are test data
        for split in ("train", "test"):
            folder = os.path.join(FIELD_ROOT, split, FIELD["folders"][class_name])
            field_records += [
                {
                    "path": os.path.join(folder, f),
                    "label": label,
                    "class_name": class_name,
                }
                for f in sorted(os.listdir(folder))
                if f.lower().endswith(IMAGE_SUFFIXES)
            ]
    field_df = pd.DataFrame(field_records)
    FIELD_IMAGES = build_cache(field_df["path"], "field")
    y_field = field_df["label"].to_numpy()
    field_present = np.unique(y_field)
    print(field_df["class_name"].value_counts().reindex(CLASSES).to_dict())

    field_preds = predict_labels(
        shift_models, {"test": TEST_IMAGES, "field": FIELD_IMAGES}
    )
    field_results, field_seed, field_boot = shift_table(
        shift_models,
        field_preds,
        "field",
        y=y_field,
        groups=np.arange(len(field_df)),  # no leaf ids: every photo is its own group
        y_clean=data.y_test,
        columns=("clean test", "field photos", "95% interval (photos)"),
    )
    field_results.to_csv(out_path("field_photos.csv"))
    shift_plot(
        field_results,
        clean_col="clean test",
        shift_col="field photos",
        seed_scores=field_seed,
        ylabel="macro-F1",
        title="Clean test and field photos (bars: mean, dots: seeds)",
        file_name="field_photos.png",
    )
    display(field_results.round(3))
else:
    print("Section 15.2 is skipped: the PlantDoc photos are not available.")

In [ ]:
if FIELD_AVAILABLE:
    # Recall per class on the field photos, mean over the seeds
    field_recall = pd.DataFrame.from_dict(
        per_class_recall(shift_models, field_preds, "field", y_field, field_present),
        orient="index",
        columns=[SHORT_NAMES[c] for c in field_present],
    )
    field_recall.to_csv(out_path("field_photos_recall.csv"))
    display(field_recall.round(3))
else:
    print("Section 15.2 is skipped: the PlantDoc photos are not available.")

In [ ]:
if FIELD_AVAILABLE:
    field_verdicts = countermeasure_verdicts(
        BASES, field_seed, field_boot, "95% interval (photos)"
    )
    field_verdicts.to_csv(out_path("field_photos_verdicts.csv"), index=False)
    display(field_verdicts.round(3))
else:
    print("Section 15.2 is skipped: the PlantDoc photos are not available.")

## 16. Optional extra: Can the models tell when they are wrong? Confidence and abstention

A model that is unsure could hand a photo to an expert instead of guessing. This works only if wrong predictions have a lower confidence than correct ones. We take the highest predicted probability as the **confidence**, sort the test photos by it and let the model **abstain** on the least confident photos. The **risk-coverage curve** shows the accuracy of the photos that are still answered (y axis) against the share of photos that are answered (x axis). A curve that rises quickly when a few photos are left out means that the confidence flags the errors. A flat curve means that the model is as confident when it is wrong as when it is right. We show the curves for the original test images and under distribution shift.

The curves are descriptive: no abstention threshold is chosen on the test set. The table also reports the **expected calibration error** (ECE), the average gap between confidence and accuracy over ten confidence bins.

In [ ]:
# The test leaves under the conditions of Section 13 (predictions are cached from notebook 04)
tv = make_test_variants(data, verbose=False)
VARIANTS, y_rob = tv.variants, tv.y
rob_probs = predict_probs(FINAL_MODELS, VARIANTS)

coverages = np.linspace(1.0, 0.2, 81)
fig, axes = plt.subplots(
    1, len(FINAL_MODELS), figsize=(5 * len(FINAL_MODELS), 3.8), sharey=True
)
for ax, name in zip(axes, FINAL_MODELS, strict=True):
    for variant in VARIANTS:
        curve = np.mean(
            [
                risk_coverage(rob_probs[(name, seed, variant)], y_rob, coverages)
                for seed in SEEDS
            ],
            axis=0,
        )
        ax.plot(
            coverages,
            curve,
            label=variant,
            linestyle="--" if variant == "original" else "-",
        )
    ax.set_title(name)
    ax.set_xlabel("share of photos answered")
    ax.invert_xaxis()
    ax.grid(alpha=0.3)
axes[0].set_ylabel("accuracy of the answered photos")
axes[-1].legend(loc="lower left", fontsize=8)
finish("risk_coverage.png")

In [ ]:
rows = []
for name in FINAL_MODELS:
    for variant in VARIANTS:
        seed_stats = []
        for seed in SEEDS:
            probs = rob_probs[(name, seed, variant)]
            wrong = probs.argmax(1) != y_rob
            least_confident = np.argsort(probs.max(1), kind="stable")[
                : int(0.05 * len(y_rob))
            ]
            seed_stats.append(
                {
                    "errors": wrong.sum(),
                    "accuracy (all photos answered)": 1 - wrong.mean(),
                    "accuracy (95% answered)": risk_coverage(
                        probs, y_rob, np.array([0.95])
                    )[0],
                    "share of errors among the 5% least confident": (
                        wrong[least_confident].sum() / wrong.sum()
                        if wrong.any()
                        else np.nan
                    ),
                    "ECE": expected_calibration_error(probs, y_rob),
                }
            )
        rows.append(
            {
                "model": name,
                "variant": variant,
                **pd.DataFrame(seed_stats).mean().to_dict(),
            }
        )
confidence_table = pd.DataFrame(rows).set_index(["model", "variant"])
confidence_table.to_csv(out_path("confidence_abstention.csv"))
confidence_table.round(3)

## 17. Discussion

**Headline.** On the held-out test set (892 images, 3 seeds) all three models reach about **99.3% macro-F1**, and their differences are smaller than the seed-to-seed variation. They differ clearly only **outside the training distribution**: with the background removed, in grayscale, and under lighting changes, blur or noise. There the pre-trained models, especially the ViT, are far more robust. The from-scratch CNN shows a **background shortcut**: on a black background it predicts "late blight" for almost every leaf (Section 13.2 separates the background from the cut-out leaf).

| Model (selected on validation) | Test accuracy | Test macro-F1 | Errors (3 seeds × 892) | Params | Size | ms/image (batch 64, MPS) |
|---|---|---|---|---|---|---|
| SE-CNN (batch 64) | 0.9944 ± 0.0011 | 0.9931 ± 0.0014 | 15 | 0.41 M | 1.7 MB | 1.9 |
| ResNet18 (two-stage + cosine, lr 3e-4) | 0.9933 ± 0.0022 | 0.9925 ± 0.0021 | 18 | 11.2 M | 45 MB | 1.6 |
| ViT-B/16 (two-stage + cosine) | 0.9948 ± 0.0017 | 0.9937 ± 0.0022 | 14 | 85.8 M | 343 MB | 12.6 |

### Data protocol (Sections 3–5)
- **One training series per class, official leaf ids only.**
  - Training, validation and the clean test use `RS_HL` (healthy), `GCREC_Bact.Sp` (bacterial spot), `RS_Erly.B` (early blight) and `RS_Late.B` (late blight): **4,674 photos in 1,169 leaf groups**, all with official leaf ids. One photo without leaf information is excluded.
  - Split by leaf groups: 3,270 train, 704 validation, 700 test photos.
  - The other series of these classes (`UF.GRC_BS_Lab`, `GH_HL`, `GHLB`, `GHLB2`, `GHLB2ES`, `GHLB_PS`, together **1,948 photos**) are never used for training or model selection. They form the series test of Section 15. Stray single files in the class folders (series with fewer than 20 photos) are ignored.
- **Leakage audit.**
  - No leaf group appears in two splits.
  - No near-duplicate pairs were found at the merge threshold (≤ 10 of 256 bits), and no pair crosses the splits at the looser audit threshold (≤ 20 bits).
  - Series without official leaf ids are not trained on, because file numbers do not identify a leaf reliably (the same greenhouse leaf is photographed on several days).
- **Background and series.**
  - Every class comes from one series, so the photo background is class-specific (Section 4). A model can therefore learn the background instead of the leaf, the question of Sections 13 and 14.
  - The 150 dark-background photos of `GHLB_PS` (late blight) are part of the series test, not of training.

### Hyperparameters and ablations (Section 9)
- **Training length mattered most.**
  - With 50 epochs, every CNN variant reaches a validation macro-F1 between **0.992 and 0.995**. The best epochs are 36–50, so the models kept improving almost to the end.
  - With 15 epochs, the same full recipe reached only 0.978.
- **The ablations show no measurable effect of any single component.** Relative to the full recipe (0.9926):

  | Change | Δ val macro-F1 |
  |---|---|
  | no SE attention | −0.0004 |
  | no augmentation | +0.0004 |
  | no dropout | +0.0014 |
  | no class weights | ±0.0000 |
  | constant LR | +0.0002 |
  | batch 64 | +0.0028 |
  | lr 3e-3 | +0.0028 |
  | lr 3e-4 | +0.0014 |

  One validation image is worth about 0.001. The three seeds of the selected CNN also vary by 0.003 (0.9954 / 0.9926 / 0.9926). So none of these differences is distinguishable from noise, and selecting "batch 64" over "lr 3e-3", tied at 0.9954, was effectively a coin toss.

  On clean, controlled PlantVillage images a well-trained small CNN is already near its ceiling. These regularisers matter more when data is scarce or distributions shift, which Section 13 tests.
- **SE attention is not worth it here.** It adds 5.7% parameters (389,924 → 412,224) and **48% training time per epoch** on MPS (21 s → 31 s), but gives no accuracy gain.
- **Fine-tuning strategy (ResNet18).**
  - Training only the head on a **frozen backbone** reaches just **0.925**. ImageNet features have to be adapted to recognise fine lesion textures.
  - All strategies that unfreeze the backbone reach 0.991–0.992. Two-stage fine-tuning with a cosine schedule (lr 3e-4: 0.9923) is marginally better than full fine-tuning at a constant learning rate (0.9906), but the gap is within noise.
  - Two-stage fine-tuning is still the safer recipe. The constant-LR run peaked early (epoch 6) and then fluctuated, while the cosine runs improved steadily until epochs 14–15.
- **The ViT** reached a lower *validation* macro-F1 (0.984) than the CNNs. Its learning curves show training accuracy reaching 100% by about epoch 6, while validation loss creeps up slightly from epoch 2. With 4 trainable blocks (~28 M parameters) it fits the 4,255 training images almost perfectly. On the test set it nevertheless scored highest, which shows how small validation and test sets reorder models that are this close.

### CNN vs. ResNet18 vs. ViT (Section 10)
- **The test differences are within one standard deviation.** Mean macro-F1 spans only 0.9925–0.9937, about one test image, while each model's seed std is 0.0014–0.0022. On clean test images the three models are **statistically indistinguishable**, so we do not claim that any of them is better on in-distribution data.
- **Per class:**
  - healthy is perfect or nearly so for every model (F1 ≥ 0.998);
  - bacterial spot is ≥ 0.995;
  - **early blight is the hardest class** for all models (F1 0.984–0.988), followed by late blight (0.989–0.991).

### Compute trade-offs (Section 11)
| | SE-CNN | ResNet18 | ViT-B/16 |
|---|---|---|---|
| Parameters | 0.41 M | 11.2 M (27×) | 85.8 M (208×) |
| Single-image latency | 2.8 ms | 4.5 ms | 16.8 ms |
| Batched inference (per image) | 1.9 ms | 1.6 ms | 12.6 ms |
| Training per run | ~26 min (50 epochs) | ~7 min (16 epochs) | ~23 min (12 epochs) |

- On these clean images the ViT's extra cost buys **no accuracy**. A mobile app for lab-like photos would choose the 1.7 MB CNN.
- ResNet18 trains fastest, because the pre-trained features converge in a few epochs.
- Section 13 changes this conclusion for real-world photos.

### Errors (Section 12)
- **Early blight ↔ late blight dominates.** It accounts for 10 of the CNN's 15 errors, 8 of ResNet18's 18 and 10 of the ViT's 14 (pooled over 3 seeds).
  - The misclassified images are mostly early-blight leaves with **large, dark necrotic patches and yellowing margins**, which look like late-blight lesions. The concentric rings typical of early blight are not visible in them.
  - The same early-blight leaf, curled with a large brown lesion, fooled all three models.
  - The next most common error is bacterial spot ↔ early/late blight, on leaves with many small dark spots.
- **Confidence.** Wrong predictions are clearly less confident than correct ones: mean 0.74 (CNN), 0.72 (ResNet18) and 0.85 (ViT), versus ≥ 0.994. A confidence threshold could therefore send uncertain cases to an expert. The ViT's errors are the most confident, so it is the hardest of the three to flag this way.
- **Grad-CAM.**
  - The CNN's heat maps are sharp and usually sit on **individual lesions**, e.g. the brown lesion at the edge of a late-blight leaf, or the spots on bacterial-spot leaves.
  - In one of its errors (early blight predicted as late blight), the heat map lies on the **background and shadow beside the leaf** rather than on the lesions. This is a direct sign of the background shortcut.
  - ResNet18's maps are coarser (7×7 resolution) but consistently centred on the diseased part of the leaf. ResNet18 classified all three of the CNN's error examples correctly.

### Robustness and background bias (Section 13)
Mean test macro-F1 over 3 seeds, on the same test leaves:

| Variant | SE-CNN | ResNet18 | ViT-B/16 |
|---|---|---|---|
| original | 0.993 | 0.993 | 0.994 |
| background removed | **0.106** | 0.454 | 0.714 |
| grayscale | 0.404 | 0.475 | 0.542 |
| darker (×0.6) | 0.772 | 0.981 | 0.988 |
| blur (σ = 2) | 0.617 | 0.431 ± 0.151 | 0.717 |
| noise (σ = 20) | 0.359 ± 0.154 | 0.472 | **0.949** |

- **Background shortcut indicated.**
  - With the background removed, the CNN predicts **late blight for 96–99% of the leaves of every class**. It behaves as if it had learned "black background ⇒ late blight" from the 150 dark-background training images.
  - ResNet18 is partly affected: 100% of healthy and 42% of early-blight leaves become late blight.
  - The ViT is the least affected (healthy 61%, early blight 36%, bacterial spot 13%).
  - Healthy leaves suffer most for every model, plausibly because they have no lesions to outweigh the background cue.
  - Caveat: segmented images are out-of-distribution in other ways too, such as sharp mask edges, and the segmentation sometimes cuts away the thin edge of the leaf. The systematic shift towards late blight points to the shortcut. Section 13.2 separates the two by changing only the background of the same cut-out leaf.
- **Colour dependence.** All models drop to 0.40–0.54 macro-F1 in grayscale, so colour (yellowing, brown necrosis) is a major cue. This is expected, since our augmentation deliberately preserves colour, but it means a different camera white balance could hurt.
- **Lighting, blur and noise.**
  - The pre-trained models are almost unaffected by darker images (≥ 0.98), while the CNN drops to 0.77: plausibly ImageNet pre-training gives lighting invariance that our ±10% brightness jitter does not.
  - Noise separates the models most. The ViT keeps 0.949, while the CNNs collapse (0.36–0.47, with high variance between seeds). A plausible explanation is that the ViT's 16×16 patch embedding averages out pixel noise, whereas small 3×3 convolution filters respond to it.
  - Blur hurts every model, and ResNet18 is the most unstable (std 0.151).
- **Conclusion for deployment.** On field photos, which have varied backgrounds, lighting and focus, the ViT would likely be the most robust of the three, in particular to noise and to the background, and the CNN the least robust to the background. Under blur the order is less clear (ResNet18 is the most unstable), so there is no single ranking.
  - The CNN's identical in-distribution accuracy hides a model that partly classifies backgrounds.
  - Likely fixes: augment with random backgrounds (paste segmented leaves onto varied backgrounds), remove or balance the dark-background series, and add noise/blur augmentation. Each should be verified with the same robustness tests. Section 14 tests the first of them.

### Limitations
- **Leaf masks.** The masks come from PlantVillage's own segmented photos. The segmentation occasionally cuts away the thin edge of a leaf (in a first look at 14 random photos 4 were affected, always only the edge and no lesion). Training backgrounds and the tests of Sections 13.2 and 14.3 use the same masks, so all models are compared on identical leaves. The countermeasure models were however also trained on cut-out leaves and the baselines were not, so part of a gain may reflect familiarity with the cut-out style. Absolute scores under a background change are approximations.
- **Countermeasure chosen after the test results.** The countermeasure of Section 14 was motivated by the test results of Section 13. With three seeds the "clearly better" rule is a cautious rule of thumb, not a significance test.
- **One split.** The models were evaluated on a single leaf-group split with small validation and test sets. Model *selection* among near-identical configurations is therefore essentially random, and the ViT's validation-to-test reversal shows this.
- **Residual leakage risk.** The official leaf map can give neighbouring ids to the same physical leaf, which a blind review of neighbouring photos found in the `RS_Erly.B` and `RS_HL` series. Our hash audit only catches near-identical photos. Test scores may be slightly optimistic.
- **Lab photos only.** There are no labelled field images. The robustness variants are synthetic or come from the same photo sessions, so they indicate generalisation but do not measure it.
- **Tuning seed reused.** Tuning used seed 42, which is also one of the three final seeds. This slightly favours the selected configurations on validation; test results are unaffected.
- **Reproducibility across hardware.** Training ran on Apple MPS, where a few operations have no deterministic implementation. Exact numbers may differ slightly on other hardware.

## References & Acknowledgements
- D. Singh, N. Jain, P. Jain, P. Kayal, S. Kumawat, N. Batra. *PlantDoc: A Dataset for Visual Plant Disease Detection.* CoDS-COMAD 2020 (field photos of Section 15.2, CC BY 4.0).
- M. A. Noyan. *Uncovering bias in the PlantVillage dataset.* arXiv:2206.04374, 2022 (preprint).
- S. P. Mohanty, D. P. Hughes, M. Salathé. *Using Deep Learning for Image-Based Plant Disease Detection.* Frontiers in Plant Science, 2016 (PlantVillage dataset).
- K. He et al. *Deep Residual Learning for Image Recognition.* CVPR 2016 (ResNet).
- A. Dosovitskiy et al. *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale.* ICLR 2021 (ViT).
- J. Hu, L. Shen, G. Sun. *Squeeze-and-Excitation Networks.* CVPR 2018.
- R. R. Selvaraju et al. *Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization.* ICCV 2017.
- I. Loshchilov, F. Hutter. *Decoupled Weight Decay Regularization* (AdamW), ICLR 2019; *SGDR: Stochastic Gradient Descent with Warm Restarts* (cosine schedule), ICLR 2017.
- Libraries: PyTorch, torchvision (pre-trained ResNet18 / ViT-B/16 weights), scikit-learn, SciPy, pandas, NumPy, Matplotlib, Pillow, Jupyter.